In [74]:
import os
import torch
import torch.nn as nn
import scipy.io as sio
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights,
    resnet18, ResNet18_Weights,
    swin_t, Swin_T_Weights,
)

In [75]:
# Paths (update to your local layout)
train_image_dir = "/home/wins057/Documents/Projects/python-for-dl-homework/week10/dataset/ShanghaiTech_Crowd_Counting_Dataset/part_A_final/train_data/images"
train_mat_dir   = "/home/wins057/Documents/Projects/python-for-dl-homework/week10/dataset/ShanghaiTech_Crowd_Counting_Dataset/part_A_final/train_data/ground_truth"
test_image_dir  = "/home/wins057/Documents/Projects/python-for-dl-homework/week10/dataset/ShanghaiTech_Crowd_Counting_Dataset/part_A_final/test_data/images"
test_mat_dir    = "/home/wins057/Documents/Projects/python-for-dl-homework/week10/dataset/ShanghaiTech_Crowd_Counting_Dataset/part_A_final/test_data/ground_truth"

In [76]:
import os
import numpy as np
import scipy.io as sio
from PIL import Image
from torch.utils.data import Dataset
import torch

class PeopleCountingDataset(Dataset):
    def __init__(self, image_dir, mat_dir, transform=None):
        self.image_dir = image_dir
        self.mat_dir = mat_dir
        self.transform = transform
        self.image_files = []
        self.counts = []

        for mat_name in sorted(os.listdir(mat_dir)):
            if not mat_name.endswith(".mat"):
                continue

            # Extract people coordinates from .mat file
            mat_path = os.path.join(mat_dir, mat_name)
            mat = sio.loadmat(mat_path)
            points = mat["image_info"][0, 0][0, 0][0]  # (N, 2) array of coordinates
            count = len(points)

            # Derive image filename from mat filename
            img_name = mat_name.replace("GT_", "").replace(".mat", ".jpg")
            self.image_files.append(img_name)
            self.counts.append(count)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(self.counts[idx], dtype=torch.float32)
        return img, label


In [100]:
# --- Data Loading Function ---
def load_data(train_images, train_mats, test_images, test_mats,
              transform, batch_size=16, num_workers=0):
    train_ds = PeopleCountingDataset(train_images, train_mats, transform)
    test_ds  = PeopleCountingDataset(test_images,  test_mats, transform)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [101]:
# --- Model Factory ---
def get_model(name):
    if name == "efficientnet_b0":
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "mobilenet_v3_large":
        m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif name == "shufflenet_v2_x1_0":
        m = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "resnet":
        weights = ResNet18_Weights.DEFAULT
        m = resnet18(weights=weights)
        m.fc =  nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    elif name == "swin":
        weights = Swin_T_Weights.DEFAULT
        m = swin_t(weights=weights)
        m.head = nn.Linear(m.head.in_features, 1) # nn.Sequential(nn.Linear(in_features, 1))
    else:
        raise ValueError(f"Unknown model: {name}")
    return m

In [102]:
# --- Checkpoint Utilities ---
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)

In [111]:
def train_and_validate(model_name, train_loader, test_loader, device,
                       num_epochs=150, ckpt_dir="checkpoints",
                       lr=0.01, patience_es=40):

    os.makedirs(ckpt_dir, exist_ok=True)
    model = get_model(model_name).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = StepLR(optimizer, step_size=30, gamma=0.2)

    best_val_mae = float("inf")
    es_counter = 0
    best_epoch = 0

    for epoch in range(1, num_epochs + 1):
        # --- Training ---
        model.train()
        train_mae = 0.0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            preds = model(imgs).view(-1)
            loss = criterion(preds, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_mae += torch.abs(preds - targets).sum().item()

        train_mae /= len(train_loader.dataset)

        # --- Validation ---
        model.eval()
        val_mae = 0.0
        with torch.no_grad():
            for imgs, targets in test_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                preds = model(imgs).view(-1)
                val_mae += torch.abs(preds - targets).sum().item()

        val_mae /= len(test_loader.dataset)
        scheduler.step()

        print(f"{model_name} | Epoch {epoch}/{num_epochs} | "
              f"Train MAE: {train_mae:.2f} | Val MAE: {val_mae:.2f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.1e}")

        # Save best model only
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            es_counter = 0
            best_epoch = epoch
            best_path = os.path.join(ckpt_dir, f"{model_name}_best.pth")
            save_checkpoint(model, optimizer, epoch, best_path)
        else:
            es_counter += 1
            if es_counter >= patience_es:
                print(f"Early stopping at epoch {epoch} (no improvement in {patience_es} epochs).")
                break

    print(f"Training finished. Best model saved from epoch {best_epoch} with MAE: {best_val_mae:.2f}")


In [116]:
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
# ])

# --- Data Augmentation ---
# Use a more aggressive data augmentation strategy
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize all images to the same size
    transforms.RandomAffine(
        degrees=15,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_loader, test_loader = load_data(
    train_image_dir, train_mat_dir, test_image_dir, test_mat_dir,
    transform, batch_size=16, num_workers=4
)

In [117]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for name in ["resnet"]:
    train_and_validate(name, train_loader, test_loader, device, num_epochs=100)

resnet | Epoch 1/100 | Train MAE: 490.44 | Val MAE: 437.65 | LR: 1.0e-02
resnet | Epoch 2/100 | Train MAE: 368.48 | Val MAE: 786.50 | LR: 1.0e-02
resnet | Epoch 3/100 | Train MAE: 301.72 | Val MAE: 286.00 | LR: 1.0e-02
resnet | Epoch 4/100 | Train MAE: 274.94 | Val MAE: 593.88 | LR: 1.0e-02
resnet | Epoch 5/100 | Train MAE: 300.14 | Val MAE: 352.71 | LR: 1.0e-02
resnet | Epoch 6/100 | Train MAE: 265.89 | Val MAE: 1645.94 | LR: 1.0e-02
resnet | Epoch 7/100 | Train MAE: 294.09 | Val MAE: 195.55 | LR: 1.0e-02
resnet | Epoch 8/100 | Train MAE: 276.46 | Val MAE: 303.68 | LR: 1.0e-02
resnet | Epoch 9/100 | Train MAE: 243.48 | Val MAE: 272.46 | LR: 1.0e-02
resnet | Epoch 10/100 | Train MAE: 250.56 | Val MAE: 692.72 | LR: 1.0e-02
resnet | Epoch 11/100 | Train MAE: 283.87 | Val MAE: 248.68 | LR: 1.0e-02
resnet | Epoch 12/100 | Train MAE: 260.81 | Val MAE: 343.52 | LR: 1.0e-02
resnet | Epoch 13/100 | Train MAE: 243.85 | Val MAE: 182.82 | LR: 1.0e-02
resnet | Epoch 14/100 | Train MAE: 243.94 | Va

In [119]:
def load_model(model_name, checkpoint_path, device):
    """
    Recreate the architecture for `model_name`, load weights from checkpoint_path,
    move to device, and set to eval mode.
    """
    model = get_model(model_name)
    ckpt  = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    return model

def evaluate_model(model_name, checkpoint_path, test_loader, device):
    """
    Load model, run on test_loader, return predictions, targets, and MAE.
    """
    model = load_model(model_name, checkpoint_path, device)
    all_preds, all_targets = [], []

    model.eval()
    with torch.no_grad():
        for imgs, targets in test_loader:
            imgs = imgs.to(device)
            preds = model(imgs).view(-1).cpu().tolist()  # safer than .squeeze()
            all_preds.extend(preds)
            all_targets.extend(targets.tolist())

    mae = sum(abs(p - t) for p, t in zip(all_preds, all_targets)) / len(all_preds)
    return all_preds, all_targets, mae

# Example usage
if __name__ == "__main__":
    for name in ["resnet"]:
        ckpt = f"checkpoints/{name}_best.pth"
        preds, targets, mae = evaluate_model(name, ckpt, test_loader, device)
        print("Test samples:", len(test_loader.dataset))
        print(f"{name} Test MAE: {mae:.2f}")
        print(f"Predictions: {preds[:5]}")   # ← should always be a list of floats
        print(f"Targets: {targets[:5]}")

Test samples: 182
resnet Test MAE: 144.21
Predictions: [572.5314331054688, 225.1663055419922, 311.1109313964844, 256.71771240234375, 267.4056091308594]
Targets: [172.0, 502.0, 391.0, 211.0, 223.0]
